# Account Statement Review

**Project 01 — AI Testing Portfolio**

A small Python program that reviews a customer's bank account statement, plus automated tests that verify it.

**What it does**
- Calculates total deposits, withdrawals, and the closing balance
- Flags suspicious transactions (above 5,000 SAR in either direction)
- Assigns a risk level: `Low`, `Review`, or `High`
- Prints a formatted review report

**How it is tested**
- Reconciliation check
- Boundary Value Analysis (BVA)
- Zero-One-Many (empty statement)
- Mutation testing

## 1. Customer data

Positive amounts are deposits; negative amounts are withdrawals (SAR).

In [1]:
customer_name = "Yousif"
opening_balance = 7500
transactions = [2000, -1500, -300, 6500, -800, -9000]

## 2. Business logic

The logic is split into small functions so that each rule can be tested on its own.

**Suspicious-transaction rule:** any single transaction above 5,000 SAR, deposit or withdrawal.

In [2]:
SUSPICIOUS_LIMIT = 5000


def calculate_closing_balance(opening_balance, transactions):
    """Return the balance after applying all transactions."""
    return opening_balance + sum(transactions)


def calculate_totals(transactions):
    """Return (total deposits, total withdrawals)."""
    deposits = 0
    withdrawals = 0
    for t in transactions:
        if t > 0:
            deposits = deposits + t
        else:
            withdrawals = withdrawals + t
    return deposits, withdrawals


def count_suspicious(transactions):
    """Count transactions above the limit in either direction."""
    count = 0
    for t in transactions:
        if t > SUSPICIOUS_LIMIT or t < -SUSPICIOUS_LIMIT:
            count = count + 1
    return count


def get_risk_level(transactions):
    """Low = 0 suspicious, Review = 1-2, High = 3 or more."""
    count = count_suspicious(transactions)
    if count == 0:
        return "Low"
    elif count <= 2:
        return "Review"
    else:
        return "High"

## 3. Review report

In [3]:
def print_report(customer_name, opening_balance, transactions):
    deposits, withdrawals = calculate_totals(transactions)
    closing_balance = calculate_closing_balance(opening_balance, transactions)

    if len(transactions) > 0:
        average = round(sum(transactions) / len(transactions), 2)
    else:
        average = 0

    reconciled = closing_balance == opening_balance + deposits + withdrawals

    print("=" * 35)
    print("   ACCOUNT STATEMENT REVIEW")
    print("=" * 35)
    print(f"Customer:          {customer_name}")
    print(f"Opening balance:   {opening_balance} SAR")
    print(f"Transactions:      {len(transactions)}")
    print(f"Total deposits:    {deposits} SAR")
    print(f"Total withdrawals: {withdrawals} SAR")
    print(f"Closing balance:   {closing_balance} SAR")
    print(f"Average txn:       {average} SAR")
    print(f"Suspicious txns:   {count_suspicious(transactions)}")
    print(f"Risk level:        {get_risk_level(transactions)}")
    print(f"Reconciliation:    {reconciled}")
    print("=" * 35)


print_report(customer_name, opening_balance, transactions)

   ACCOUNT STATEMENT REVIEW
Customer:          Yousif
Opening balance:   7500 SAR
Transactions:      6
Total deposits:    8500 SAR
Total withdrawals: -11600 SAR
Closing balance:   4400 SAR
Average txn:       -516.67 SAR
Suspicious txns:   2
Risk level:        Review
Reconciliation:    True


## 4. Automated tests

| Test | Technique | Why it matters |
|------|-----------|----------------|
| Closing balance of the sample statement | Reconciliation | The balance must match opening balance plus all movements |
| Empty statement | Zero-One-Many | A new customer with no transactions must not crash the program |
| `5000` is **not** suspicious | BVA (on the boundary) | Catches `>` accidentally written as `>=` |
| `5001` and `-5001` are suspicious | BVA (just above the boundary) | Checks both directions |
| One test per risk level | Equivalence partitioning | Each output class is covered |

In [4]:
# Reconciliation
assert calculate_closing_balance(7500, [2000, -1500, -300, 6500, -800, -9000]) == 4400, "Sample statement should close at 4400"
assert calculate_totals([2000, -1500, 6500]) == (8500, -1500), "Totals should split deposits and withdrawals"

# Zero-One-Many
assert calculate_closing_balance(7500, []) == 7500, "Empty statement should keep the opening balance"
assert count_suspicious([]) == 0, "Empty statement has no suspicious transactions"

# Boundary Value Analysis
assert count_suspicious([5000]) == 0, "5000 is on the limit and should NOT be suspicious"
assert count_suspicious([-5000]) == 0, "-5000 is on the limit and should NOT be suspicious"
assert count_suspicious([5001]) == 1, "5001 deposit should be suspicious"
assert count_suspicious([-5001]) == 1, "5001 withdrawal should be suspicious"

# Risk levels
assert get_risk_level([]) == "Low"
assert get_risk_level([6000, -7000]) == "Review"
assert get_risk_level([6000, -7000, 8000]) == "High"

print("All tests passed ✅")

All tests passed ✅


## 5. Mutation testing

To check that the tests are strong enough, the rule was deliberately broken by changing `>` to `>=`:

```python
if t >= SUSPICIOUS_LIMIT or t < -SUSPICIOUS_LIMIT:
```

**Result:** the BVA test `count_suspicious([5000]) == 0` failed with an `AssertionError`, so the defect was caught.

Without a test on the exact boundary value, this change would have passed unnoticed.

## 6. Open question for the business

The requirement says transactions **above** 5,000 SAR are suspicious. It does not say whether a transaction of **exactly** 5,000 SAR should be flagged.

This notebook assumes *above* (`>`). In a real project, this ambiguity would be raised with the business analyst before sign-off, because it decides whether the code should use `>` or `>=`.

## What I learned

- Variables, data types, lists, loops, conditions, and functions in Python
- Splitting logic into small functions makes each rule testable
- `assert` turns manual test cases into automated checks
- Boundary values catch defects that typical values miss
- Mutation testing shows whether the tests themselves are effective

**Next:** Project 02 applies the same testing mindset to a real fraud dataset with Pandas and a machine learning model.